In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Layered data validation and processing demonstration

## Summary

Demonstrate a workflow of a layered data validation and processing of WTG data.

## Results

### Key results

- Basic structure is a medaillon architecture w/ layers raw, bronze, silver and gold.
- Core functionality is a tracking, logging and reporting mechanism of the results, currently as structured summary, but opening the possibility for a reporting system.
- Validations are configurable and versionable.

### Details

- Layers (concept):
  - Raw:
    - Formal diagnostics for a standardisation.
  - Bronze:
    - Domain-related diagnostics for outlier detection.
  - Silver:
    - Gap filling.
  - Gold:
    - Advanced stuff.

## Remarks


## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

import ergaleiothiki
from phoibe.layered.application.config import ValidationConfig
from phoibe.layered.application.context import ValidationContext
from phoibe.layered.application.factory import ValidatorFactory, RuleRegistry
from phoibe.layered.infrastructure.io import YAMLReportRepository
from phoibe.layered.core.entities import RuleExecutionResult, Status, Severity

import phoibe.synthetic_data.turbine
from phoibe.synthetic_data.turbine import MessUpPipeline, create_default_messup_pipeline

import phoibe.layered.rules.rules_index
import phoibe.layered.rules.rules_columns
import phoibe.layered.rules.rules_values

## Provision

### Layer rule configuration

In [ ]:
VARIABLE_PATTERNS = {
    "power_kw": [r"power", r"production"],
    "wind_speed": [r"wind.*speed", r"wind.*speed.*avg"],
    "wind_speed_max": [r"wind.*speed.*max"],
    "datetime": [r"datetime", r"timestamp"],
    "date": [r"date"],
    "time": [r"time"],
}
RULES_RAW = [
    {"name": "required_variable", "params": {"variable_name": "power_kw"}},
    {"name": "temporal_attributes", "params": {}},
    {"name": "data_gaps", "params": {"good_threshold": 0.03, "acceptable_threshold": 0.1}},
    {"name": "availability", "params": {"good_threshold": 0.9, "acceptable_threshold": 0.8, "locale": "de_DE"}},
]
RULES_BRONZE = [
    {"name": "ranges", "params": {"variable_ranges": {"wind_speed": (0, 60)}}},
]

config_raw = ValidationConfig(layer_name="raw", variable_patterns=VARIABLE_PATTERNS, rules=RULES_RAW)
config_bronze = ValidationConfig(layer_name="bronze", variable_patterns=VARIABLE_PATTERNS, rules=RULES_BRONZE)

In [ ]:
RuleRegistry().list_rules()

### Synthetic data

In [ ]:
TIME = phoibe.synthetic_data.turbine.Time(start="2027-01-01", freq="10min", periods=6 * 24 * 365)
A, k = 9.7, 2.3
df = phoibe.synthetic_data.turbine.generate_wtg_scada(
    A=A, k=k, time=TIME, wtg_type=phoibe.synthetic_data.turbine.DEFAULT_WTG
)

mess_up_pipeline = phoibe.synthetic_data.turbine.create_default_messup_pipeline(incidence=23, level=13)
data = mess_up_pipeline.apply(df=df)

display(pd.concat([df.mean(), data.mean()], keys=["synthetic", "messed"], axis=1))
display(pd.concat([df.describe(), data.describe()], keys=["synthetic", "messed"], axis=1))

In [ ]:
data = data.reset_index()
data

## Validation-Transformation

### Validation of layer raw

In [ ]:
validator_raw = ValidatorFactory().create_from_memory(config=config_raw, data=data)
report_raw = validator_raw.validate(file_path=".", turbine_id="WEA 01")

### Reporting

In [ ]:
report_raw.rule_execution_results

In [ ]:
YAMLReportRepository().save(reported=[report_raw], output_path="reports/report_raw.yaml")

with open("reports/report_raw.yaml", "r") as stream:
    print(yaml.safe_dump(yaml.safe_load(stream), explicit_start=True))

In [ ]:
context = ValidationContext(
    detected_variables=validator_raw.variable_detector.detect(data), turbine_id="WEA 02", layer_name="raw"
)
context.detected_variables

### Transform layer raw

In [ ]:
key_mapping = {
    value: key for key, value in context.detected_variables.items() if key in ["datetime", "wind_speed", "power_kw"]
}
data_bronze = data.rename(columns=key_mapping)
plt.scatter(data_bronze["wind_speed"], data_bronze["power_kw"])

In [ ]:
((data_bronze.loc[:, "wind_speed"] < 0) | (data_bronze.loc[:, "wind_speed"] > 60)).sum()

data_bronze.loc[data_bronze.loc[:, "wind_speed"] > data_bronze.loc[:, "wind_speed"].quantile(0.01), "wind_speed"].min()

# data_bronze.loc[:, "wind_speed"].sort_values().hist(bins=60, log=True)
data_bronze.loc[:, "wind_speed"].sort_values().diff().reset_index().loc[:100, "wind_speed"].plot(logy=True)
data_bronze.loc[:, "wind_speed"].sort_values().reset_index().loc[:100, "wind_speed"].abs().plot(logy=True)
plt.show()
# heuristic: watch for gaps in the histogram - they are candidates for
occupation, value = np.histogram(data_bronze.loc[:, "wind_speed"], bins=60)
counts = pd.Series(index=(value[1:] + value[:-1]) / 2, data=occupation)
(counts.diff() / counts).abs().plot(logy=True)
counts.tail(20)

In [ ]:
mask_curtailments = data_bronze.loc[:, "wind_speed"] > 15
data_bronze.loc[mask_curtailments, "power_kw"].hist(bins=60, log=True)
# heuristic: for wind_speeds > 15 m/s, determine clusters of powers in histogram
# by comparing bin occupations - there should be a gap of magnitude larger than occupation in bulk.
occupation, value = np.histogram(data_bronze.loc[mask_curtailments, "power_kw"], bins=60)
counts = pd.Series(index=(value[1:] + value[:-1]) / 2, data=occupation).sort_values(ascending=False)
-counts.diff() / counts

In [ ]:
validator_bronze = ValidatorFactory().create_from_memory(config=config_bronze, data=data_bronze)
report_bronze = validator_bronze.validate(file_path=".", turbine_id="WEA 01")

In [ ]:
YAMLReportRepository().save(reported=[report_bronze], output_path="reports/report_bronze.yaml")

with open("reports/report_bronze.yaml", "r") as stream:
    print(yaml.safe_dump(yaml.safe_load(stream), explicit_start=True))